# Sequence predictors

> The gain predicted from the link's history read in order: an LSTM or a GRU over the past gains, or over the past
> steps (each one's gain, the geometry between the agents then, and their moves), next to the gradient-boosted trees.

In [ ]:
#| default_exp comm.sequence

In [ ]:
#| hide
from nbdev.showdoc import *

`comm.power_control.fit_predictor` fits gradient-boosted trees on `comm.channel_data.features`: the geometry between
the transmitter and a receiver cell (log-distance, distance, walls crossed), the link's last `L` gains (dB, most recent
first) and whether either agent moved. The trees see the past gains as `L` unrelated numbers. Here a recurrent network
reads the past in order, oldest first, in one of two ways:

* **the past gains** (`S=None`): each step is its gain, as in the features;
* **the past steps** (`S`, `build_dataset(..., steps=True)`): each step is its `step_features`: its gain, the geometry
  between the two agents at that step (where each one was), and whether each had moved since the step before. The
  sequence then knows what the gains were measured over, not only their values.

Either way, the network's last state, the geometry to the receiver's cell, the moves and the last gain go through an
MLP, which predicts the change from the last gain (dB). The loss is the pinball loss for a quantile of the gain in dB
(`quantile=0.05`: a cautious estimate, as the saved power control's), or the squared error for its mean
(`quantile=None`).

`fit_sequence_predictor` returns a `SequencePredictor`, which has a scikit-learn regressor's `predict(X)` on the same
features (a per-step one: `predict(X, S)`, and `steps` is true). `predict_gain`, `predict_on`, `LearnedChannelInversion`
(which rebuilds the past steps from the rounds it has seen), the calibration of `κ` and `τ`, `save_power_control` and
`load_power_control` take it as they take the trees. `fit_model` fits any of the three kinds, and `predictor_name` names
them in `comm.evaluation`'s comparisons (`fit_baselines(..., extra=...)`).

A longer history is the data built with more past gains (`build_dataset(..., history=L)`); `with_history` keeps the most
recent ones, so several history lengths are compared on the same transmissions.

In [ ]:
#| export
from __future__ import annotations

import copy
import math

import numpy as np

from c3jepa_wm.comm.channel_data import STEP_FEATURES
from c3jepa_wm.comm.power_control import fit_predictor

GEOMETRY, MOVES = 3, 2                                              # `features`: the geometry, the past gains, the moves

## The network

In [ ]:
#| export
def _net(history: int, cell: str, steps: bool, hidden: int, layers: int, head: int, dropout: float):
    import torch
    from torch import nn

    class GainSequenceNet(nn.Module):
        """The past (oldest first) through an LSTM or a GRU: each step's gain, or with `steps` its `step_features`; the
        last state, the geometry to the receiver's cell, the moves and the last gain through an MLP, which predicts the
        change from the last gain."""

        def __init__(self):
            super().__init__()
            rnn = {'lstm': nn.LSTM, 'gru': nn.GRU}[cell]
            self.rnn = rnn(STEP_FEATURES if steps else 1, hidden, layers, batch_first=True,
                           dropout=dropout if layers > 1 else 0.0)
            self.head = nn.Sequential(nn.Linear(hidden + GEOMETRY + MOVES + 1, head), nn.ReLU(),
                                      nn.Linear(head, head), nn.ReLU(), nn.Linear(head, 1))

        def forward(self, x, s=None):                               # x: (B, F), s: (B, L, STEP_FEATURES), standardized
            last = x[:, GEOMETRY]
            past = s if steps else x[:, GEOMETRY:GEOMETRY + history].unsqueeze(-1)
            out, _ = self.rnn(past.flip(1))                         # oldest first
            static = torch.cat([x[:, :GEOMETRY], x[:, -MOVES:], last[:, None]], 1)
            return last + self.head(torch.cat([out[:, -1], static], 1)).squeeze(1)

    return GainSequenceNet()


class SequencePredictor:
    """A fitted `GainSequenceNet` as a scikit-learn regressor: `predict(X)` (with `steps`, `predict(X, S)`) -> the gain in
    dB. Large batches run on the GPU if there is one; it pickles to the CPU."""

    def __init__(self, config: dict, scale: dict, state: dict, curve: list, quantile: float | None = None,
                 importance: dict | None = None):
        self.config, self.scale, self.curve, self.quantile = config, scale, curve, quantile
        self.importance = importance                                # SI's per-weight importance over the training (`comm.continual`)
        self._build(state)

    @property
    def steps(self) -> bool:
        "Whether it reads the past steps (`S`), not only the past gains."
        return self.config['steps']

    def _build(self, state):
        import torch
        self._torch, self._gpu = torch, None
        self.net = _net(**self.config)
        self.net.load_state_dict(state)
        self.net.eval()

    def scaled(self, X: np.ndarray, S: np.ndarray | None = None) -> tuple:
        "The standardized features and past steps (float32 arrays), as the network reads them."
        c = self.scale
        Z = (np.asarray(X, np.float32) - c['mean']) / c['std']
        Zs = (np.asarray(S, np.float32)[:, :self.config['history']] - c['step_mean']) / c['step_std'] if self.steps else None
        return Z, Zs

    def target(self, gain: np.ndarray) -> np.ndarray:
        "The gains |h|² as the network predicts them: in dB, standardized."
        c = self.scale
        return ((10 * np.log10(np.maximum(np.asarray(gain, float), 1e-30)) - c['mean'][GEOMETRY]) / c['std'][GEOMETRY]).astype(np.float32)

    def loaded(self, state: dict):
        "Its network's weights replaced (by a continual update): on the CPU, the GPU copy made again when needed."
        self.net.load_state_dict({k: v.detach().cpu() for k, v in state.items()})
        self.net.eval()
        self._gpu = None

    def predict(
        self,
        X: np.ndarray, # Features (`comm.channel_data.features`)
        S: np.ndarray | None = None, # Each row's past steps (`build_dataset(..., steps=True)`'s `S`): needed with `steps`
        batch_size: int = 65536,
    ) -> np.ndarray:
        "The gain (dB) of each row."
        torch, c = self._torch, self.scale
        if self.steps and S is None:
            raise ValueError('a per-step predictor needs the past steps (S): build_dataset(..., steps=True)')
        net = self.net
        if len(X) >= 4096 and torch.cuda.is_available():            # small batches (a transmission's): the CPU
            if self._gpu is None:
                self._gpu = copy.deepcopy(self.net).cuda()
            net = self._gpu
        device = next(net.parameters()).device
        Z, Zs = self.scaled(X, S)
        out = []
        with torch.no_grad():
            for i in range(0, len(Z), batch_size):
                s = torch.from_numpy(Zs[i:i + batch_size]).to(device) if self.steps else None
                out.append(net(torch.from_numpy(Z[i:i + batch_size]).to(device), s).cpu().numpy())
        y = np.concatenate(out) if out else np.zeros(0, np.float32)
        return (y * c['std'][GEOMETRY] + c['mean'][GEOMETRY]).astype(float)   # the target shares the last gain's scale

    def __getstate__(self):
        return dict(config=self.config, scale=self.scale, curve=self.curve, quantile=self.quantile, importance=self.importance,
                    state={k: v.detach().cpu() for k, v in self.net.state_dict().items()})

    def __setstate__(self, s):
        self.config, self.scale, self.curve = s['config'], s['scale'], s['curve']
        self.quantile, self.importance = s.get('quantile'), s.get('importance')
        self._build(s['state'])


def gain_loss(quantile: float | None):
    "The training loss on standardized dB: the pinball loss of `quantile`, or the squared error (None: the mean)."
    if quantile is None:
        return lambda p, y: ((p - y) ** 2).mean()
    import torch
    return lambda p, y: torch.maximum(quantile * (y - p), (quantile - 1) * (y - p)).mean()

In [ ]:
show_doc(SequencePredictor.predict)

## Fitting

The features are standardized on the training samples, every past gain on the last gain's scale (so the sequence
keeps its shape), and the target the same way; the past steps too, their gains on that same scale. Training stops when
the loss on held-out samples has not improved for `patience` epochs, and the best epoch's weights are kept. With
`groups` (each sample's layout, `build_dataset`'s `layout`), the held-out samples are whole layouts, as the predictor
will meet new layouts.

In [ ]:
#| export
def fit_sequence_predictor(
    X: np.ndarray, # Features to the receiver's actual cell (`build_dataset`'s `X`)
    gain: np.ndarray, # The gains |h|²
    quantile: float | None = 0.05, # The quantile of the gain (dB) predicted (pinball loss); None: the mean (squared error)
    cell: str = 'lstm', # 'lstm' or 'gru'
    S: np.ndarray | None = None, # The past steps (`build_dataset(..., steps=True)`'s `S`): the per-step model; None: the past gains only
    hidden: int = 64, # Units of the recurrent cell
    layers: int = 1, # Recurrent layers
    head: int = 64, # Units of the MLP's two hidden layers
    dropout: float = 0.0, # Between recurrent layers
    epochs: int = 60, # At most
    batch_size: int = 512,
    lr: float = 2e-3,
    weight_decay: float = 1e-5,
    patience: int = 8, # Epochs without a better validation loss before stopping (the best epoch's weights are kept)
    groups: np.ndarray | None = None, # Each sample's layout: whole layouts are held out to stop on (None: random samples)
    val_fraction: float = 0.15, # Of the layouts (or samples) held out
    device: str = 'auto', # 'cpu', 'cuda', or 'auto'
    seed: int = 0,
    verbose: bool = False,
    si_xi: float = 0.1, # SI's damping: the importance of each weight over this training is kept (`importance`, for `comm.continual`)
) -> SequencePredictor:
    "An LSTM or a GRU predicting the gain in dB from the past gains (or, with `S`, the past steps), the geometry and the moves."
    import torch
    if cell not in ('lstm', 'gru'):
        raise ValueError(f"cell: 'lstm' or 'gru', not {cell!r}")
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    device = ('cuda' if torch.cuda.is_available() else 'cpu') if device == 'auto' else device
    X = np.asarray(X, np.float32)
    y = (10 * np.log10(np.maximum(np.asarray(gain, float), 1e-30))).astype(np.float32)
    history, steps = X.shape[1] - GEOMETRY - MOVES, S is not None
    if steps:
        S = np.asarray(S, np.float32)
        if S.shape[:2] != (len(X), history) or S.shape[2] != STEP_FEATURES:
            raise ValueError(f'S: ({len(X)}, {history}, {STEP_FEATURES}) for these features, not {S.shape}')
    if groups is not None:
        ids = np.unique(groups)
        val = np.isin(groups, rng.choice(ids, max(1, int(round(val_fraction * len(ids)))), replace=False))
    else:
        val = rng.random(len(y)) < val_fraction
    mean, std = X[~val].mean(0), X[~val].std(0) + 1e-6
    past = slice(GEOMETRY, GEOMETRY + history)                      # every past gain on the last one's scale
    mean[past], std[past] = mean[GEOMETRY], std[GEOMETRY]
    scale = dict(mean=mean, std=std)
    if steps:
        flat = S[~val].reshape(-1, STEP_FEATURES)
        step_mean, step_std = flat.mean(0), flat.std(0) + 1e-6
        step_mean[0], step_std[0] = mean[GEOMETRY], std[GEOMETRY]   # the steps' gains too
        scale.update(step_mean=step_mean, step_std=step_std)
    Z = torch.from_numpy((X - mean) / std)
    Zs = torch.from_numpy((S - scale['step_mean']) / scale['step_std']) if steps else None
    t = torch.from_numpy((y - mean[GEOMETRY]) / std[GEOMETRY])
    config = dict(history=history, cell=cell, steps=steps, hidden=hidden, layers=layers, head=head, dropout=dropout)
    net = _net(**config).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=max(1, patience // 3))
    loss_fn = gain_loss(quantile)
    start = {k: p.detach().clone() for k, p in net.named_parameters()}
    path = {k: torch.zeros_like(p) for k, p in net.named_parameters()}   # SI: Σ g² η over the training
    split = lambda a, m: a[torch.from_numpy(m)].to(device) if a is not None else None
    (Ztr, Zstr, ttr), (Zva, Zsva, tva) = ([split(a, m) for a in (Z, Zs, t)] for m in (~val, val))
    best, best_state, waited, curve = math.inf, None, 0, []
    for epoch in range(epochs):
        net.train()
        order = torch.randperm(len(ttr), device=device)
        for i in range(0, len(order), batch_size):
            b = order[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(net(Ztr[b], Zstr[b] if steps else None), ttr[b])
            loss.backward()
            eta = opt.param_groups[0]['lr']
            for k, p in net.named_parameters():
                if p.grad is not None:
                    path[k] += p.grad.detach() ** 2 * eta
            opt.step()
        net.eval()
        with torch.no_grad():
            v = float(loss_fn(net(Zva, Zsva), tva)) if len(tva) else float(loss)
        curve.append(v)
        sched.step(v)
        if verbose:
            print(f'epoch {epoch}: validation loss {v:.4f}')
        if v < best - 1e-5:
            best, best_state, waited = v, {k: x.detach().cpu().clone() for k, x in net.state_dict().items()}, 0
        else:
            waited += 1
            if waited >= patience:
                break
    importance = {k: (path[k] / ((best_state[k].to(device) - start[k]) ** 2 + si_xi)).cpu() for k in path}
    return SequencePredictor(config, scale, best_state, curve, quantile, importance)

In [ ]:
show_doc(fit_sequence_predictor)

## Any of the predictors

In [ ]:
#| export
KINDS = ('trees', 'lstm', 'gru')

def predictor_name(kind: str, quantile: float | None, steps: bool = False) -> str:
    "The name a predictor goes by in the comparisons (the trees' mean: `comm.evaluation`'s 'geometry + history mean')."
    what = 'quantile' if quantile is not None else 'mean'
    if kind == 'trees':
        return 'geometry + history mean' if quantile is None else 'geometry + history quantile (trees)'
    return f"{kind.upper()}, {'per-step geometry' if steps else 'geometry'} + history {what}"

def fit_model(
    train: dict, # The training samples (`build_dataset`'s; with `S` for a per-step model)
    kind: str = 'trees', # 'trees' (`fit_predictor`), 'lstm' or 'gru' (`fit_sequence_predictor`)
    quantile: float | None = 0.05, # The quantile of the gain (dB) predicted; None: the mean
    steps: bool = False, # lstm, gru: over the past steps (`S`), not only the past gains
    trees: dict | None = None, # Other options of `fit_predictor` (max_iter, learning_rate, ...)
    sequence: dict | None = None, # Other options of `fit_sequence_predictor` (hidden, epochs, device, ...)
):
    "A gain predictor of `kind`, fit on `train`: what `predict_gain` and `LearnedChannelInversion` take."
    if kind not in KINDS:
        raise ValueError(f'kind: one of {KINDS}, not {kind!r}')
    if kind == 'trees':
        if steps:
            raise ValueError('the trees read the features only (steps=False)')
        return fit_predictor(train['X'], train['gain'], quantile, **(trees or {}))
    if steps and 'S' not in train:
        raise ValueError('a per-step model needs the past steps: build_dataset(..., steps=True)')
    return fit_sequence_predictor(train['X'], train['gain'], quantile, kind, train['S'] if steps else None,
                                  groups=train.get('layout'), **(sequence or {}))

In [ ]:
show_doc(fit_model)

## Example

The stand-in channel of `comm.evaluation` (free-space loss and 6 dB per wall crossed, Rayleigh fading on top), built
with 8 past steps. Each predictor at 4 and 8 past gains, each with its own `κ` and `τ` for 95% chosen on the
calibration layouts.

In [ ]:
import pickle
import tempfile
from pathlib import Path

from c3jepa_wm.comm.channel_data import (PairCache, build_dataset, collect_trajectories, trace_visited_pairs, wall_geometry,
                                         with_history)
from c3jepa_wm.comm.evaluation import (LEARNED, compare_network, evaluate_power_control, fit_baselines, operating_points,
                                       prediction_errors)
from c3jepa_wm.comm.power_control import Budget, calibrate, load_power_control, predict_on, save_power_control

In [ ]:
env_cfg = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 9, 'num_obstacles': 0, 'max_steps': 60}
tmp = Path(tempfile.mkdtemp())

def stand_in(walls):
    "A tracer for `PairCache`: free-space loss at 2.4 GHz (2 m cells) and 6 dB per wall crossed."
    def tracer(tx, rxs):
        return [0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20) for d, w in (wall_geometry(walls, tx, rx) for rx in rxs)]
    return tracer

SEEDS = dict(train=range(0, 24), calibration=range(24, 32), test=range(32, 40))
collect_trajectories(env_cfg, range(40), tmp, episodes=5, max_steps=60)
for seed in range(40):
    trace_visited_pairs(env_cfg, tmp, [seed], tmp, tracer=stand_in(PairCache(tmp / 'unused.npz', env_cfg, seed).walls), progress=False)
data = {k: build_dataset(tmp, tmp, s, history=8, fading='rayleigh', steps=True) for k, s in SEEDS.items()}
budget = Budget(snr_target_db=10, noise_dbm=-94, max_power_dbm=10)
print({k: len(d['gain']) for k, d in data.items()})

In [ ]:
small = dict(hidden=32, head=32, epochs=15)
specs = {name: s for s in [dict(kind='trees', trees=dict(max_iter=100)), dict(kind='trees', quantile=None, trees=dict(max_iter=100)),
                           *[dict(kind=c, steps=st, quantile=q, sequence=small) for c in ('lstm', 'gru') for st in (False, True)
                             for q in (0.05, None)]]
         for name in [predictor_name(s['kind'], s.get('quantile', 0.05), s.get('steps', False))]}
fits = {(name, L): (d, fit_model(d['train'], **s)) for L in (4, 8)
        for d in [{k: with_history(v, L) for k, v in data.items()}] for name, s in specs.items()}
print(f"{'':48s}  bias  spread  cover |  95%: reached  power")
for (name, L), (d, m) in fits.items():
    cal, test = predict_on(m, d['calibration']), predict_on(m, d['test'])
    e = prediction_errors(test, d['test']['gain'])
    o = operating_points((cal, d['calibration']['gain']), (test, d['test']['gain']), budget.g_min, (0.95,))[0.95]
    print(f"{name + f', L={L}':48s} {e['bias_db']:+5.1f}  {e['spread_db']:5.1f}  {e['coverage']:5.1%} |  {o['reliability']:12.1%}  {o['power']:4.2f}×")

In [ ]:
d4, m = fits[('LSTM, per-step geometry + history quantile', 4)]
assert m.steps and not fits[('GRU, geometry + history mean', 4)][1].steps
y = m.predict(d4['train']['X'][:100], d4['train']['S'][:100])
assert y.shape == (100,) and np.isfinite(y).all()
assert np.allclose(m.predict(d4['train']['X'][:100], data['train']['S'][:100]), y)   # only the last `history` steps are read
assert prediction_errors(predict_on(m, d4['test']), d4['test']['gain'])['coverage'] > 0.8   # a low quantile rarely overestimates
_, mean = fits[('GRU, geometry + history mean', 4)]
assert abs(prediction_errors(10 ** (mean.predict(d4['test']['X']) / 10), d4['test']['gain'])['bias_db']) < 2
assert np.allclose(pickle.loads(pickle.dumps(m)).predict(d4['train']['X'][:100], d4['train']['S'][:100]), y)
for bad in (lambda: m.predict(d4['train']['X'][:10]), lambda: fit_model(d4['train'], 'rnn'),
            lambda: fit_model(d4['train'], 'trees', steps=True),
            lambda: fit_model({k: v for k, v in d4['train'].items() if k != 'S'}, 'gru', steps=True)):
    try:
        bad()
        raise AssertionError('no error')
    except ValueError:
        pass

### Through a network

A per-step LSTM saved as the power control, and the other recurrent predictors in the comparison
(`fit_baselines(..., extra=...)`). Through C3-WM's link as comm-core blocks, `LearnedChannelInversion` rebuilds each
link's past steps from the rounds it has seen, and the network reaches what the analysis computes from the data.

In [ ]:
d = {k: with_history(v, 4) for k, v in data.items()}
model = fit_model(d['train'], 'lstm', 0.05, steps=True, sequence=small)
chosen = calibrate(predict_on(model, d['calibration']), d['calibration']['gain'], budget.g_min, target=0.95)
save_power_control(tmp / 'power_control.pkl', model, 4, chosen, budget)
saved = load_power_control(tmp / 'power_control.pkl')
extra = [dict(kind='gru', steps=True, sequence=small), dict(kind='lstm', sequence=small), dict(kind='trees', trees=dict(max_iter=100))]
fitted = fit_baselines(d['train'], 4, max_iter=100, extra=extra)
results = evaluate_power_control(saved, d, targets=(0.95,), max_iter=100, fitted=fitted)
network = compare_network(saved, d, tmp, tmp, SEEDS['test'][:3], comparison=results, target=0.95, fitted=fitted, fading='rayleigh',
                          skip=8)   # the data's first 8 steps are not in it
print('\n'.join(f"{n:55s} network {r['network']['reached']:.3f} {r['network']['power']:6.2f}×   analysis "
                 f"{r['analysis']['reached']:.3f} {r['analysis']['power']:6.2f}×" for n, r in network.items()))

In [ ]:
names = ['GRU, per-step geometry + history quantile', 'LSTM, geometry + history quantile', 'geometry + history quantile (trees)']
assert set(fitted['extra']) == set(names) and set(names) <= set(results['predictors'])
assert {f'{n}, κ, τ' for n in [LEARNED, *names]} <= set(network)
for name, r in network.items():                                            # the network is the analysis: the same past steps
    n, a = r['network'], r['analysis']
    assert abs(n['reached'] - a['reached']) < 0.005, name
    assert abs(n['power'] - a['power']) < 0.01 * a['power'], name
    assert abs(n['withheld'] - a['withheld']) < 0.005, name